In [3]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [4]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

True

In [5]:
ls_client = Client()

In [6]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [7]:

list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'Among the audio options, the Raymate Bluetooth speaker has the longest listed battery life at more than 1000 minutes of playtime. The Wekily earbuds and Jesebang earbuds each offer up to 40 hours total with their charging cases, the Pamu earbuds offer up to 30 hours total, and the MUSICOZY headband offers 20+ hours on a charge.',
 'reference_context_ids': ['B0BRV544MV',
  'B09X9838WY',
  'B09TFM1SFQ',
  'B0CFHWF326',
  'B0C996WY16'],
 'reference_descriptions': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a 

In [8]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'Which audio products have the longest battery life for travel or all-day use?'}

In [9]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

In [10]:
qdrant_client = QdrantClient(url="http://localhost:6333")

RAG PIPELINE

In [12]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [13]:
rag_pipeline("Can I get a charger?")

{'answer': 'Yes. We have chargers in stock, including:\n\n1) iPhone Charger 6ft 3Pack (Apple MFi Certified Lightning Cables) – supports fast charging up to 3A and data sync for many iPhone and iPad models.\n2) USB C to USB C Cable (INIU, 6.6ft, 100W PD) – for fast charging compatible USB-C devices like phones, tablets, and many USB-C laptops.\n\nAlso, there’s a notebook replacement power adapter listed (White-60W), but it notes that it’s the second generation and you need to check the specific Mac notebook model before buying.\n\nIf you tell me which device you’re charging (iPhone model, or USB-C laptop/phone model), I can point you to the best option.',
 'question': 'Can I get a charger?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BXC72RLD'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iPhone Charger Cord High Speed Data Sync Cable Compatible iPhone 14 13 12 11 Pro Max XS X

RAGAS Metrics

In [14]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

C:\Users\Owner\AppData\Local\Temp\ipykernel_14796\3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\Owner\AppData\Local\Temp\ipykernel_14796\3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\Owner\AppData\Local\Temp\ipykernel_14796\3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use

In [15]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\Owner\AppData\Local\Temp\ipykernel_14796\840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
C:\Users\Owner\AppData\Local\Temp\ipykernel_14796\840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [16]:
reference_input

{'question': 'Which audio products have the longest battery life for travel or all-day use?'}

In [17]:
reference_output

{'ground_truth': 'Among the audio options, the Raymate Bluetooth speaker has the longest listed battery life at more than 1000 minutes of playtime. The Wekily earbuds and Jesebang earbuds each offer up to 40 hours total with their charging cases, the Pamu earbuds offer up to 30 hours total, and the MUSICOZY headband offers 20+ hours on a charge.',
 'reference_context_ids': ['B0BRV544MV',
  'B09X9838WY',
  'B09TFM1SFQ',
  'B0CFHWF326',
  'B0C996WY16'],
 'reference_descriptions': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a 

In [18]:
result = rag_pipeline(reference_input["question"])

In [19]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [20]:
await ragas_context_precision_id_based(result, reference_output)

0.6

In [21]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [22]:
await ragas_context_recall_id_based(result, reference_output)

0.6

In [23]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [24]:
await ragas_faithfulness(result, reference_output)

0.5454545454545454

In [25]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [26]:
await ragas_relevancy(result, reference_output)

np.float64(0.8660532802447197)